In [ ]:
!pip install -qU langgraph langchain langchain-google-genai langchain-openai

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-xxx"
os.environ["GEMINI_API_KEY"] = "AIzxxx"

# Dynamic Interrupt
[Interrupt 공식 문서](https://image.png)

## Basic

### Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

### State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    context: dict  # 수집된 정보를 저장할 공간

### Node 정의

In [ ]:
from langchain.messages import SystemMessage
from langgraph.types import interrupt

def agent_node(state: ChatState):
    # 1. Context에서 정보 확인
    # state에 context 키가 없으면 빈 딕셔너리 생성
    ctx = state.get("context", {})
    food_category = ctx.get("food_category")

    # 2. [Dynamic Interrupt] 정보가 없으면 멈추고 물어봄
    if not food_category:
        print("⏸️ [System] 음식 종류 정보 부재. 사용자에게 질의(Interrupt) 시도...")

        # 실행 중단(Suspend)


    # 3. [LLM Generation] 확보된 정보로 실제 추천 생성
    system_prompt = SystemMessage(content=f"""
    당신은 서울의 맛집 전문가입니다.
    사용자가 원하는 카테고리인 '{food_category}'에 맞춰서
    실제로 유명한 맛집 1곳을 추천하고, 추천 이유를 2문장으로 설명해주세요.
    """)

    # 기존 대화 내역 + 시스템 프롬프트 조합
    messages = [system_prompt] + state["messages"]
    response = model.invoke(messages)

    # 4. 결과 반환 (메시지 추가 및 Context 업데이트)
    return {
        "messages": [response],
        "context": {"food_category": food_category} # 다음 턴을 위해 정보 저장
    }

### 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("chatbot", agent_node)
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)

In [ ]:
# InMemorySaver

In [ ]:
app = workflow.compile()

In [ ]:
app

In [ ]:
config_1 = {"configurable": {"thread_id": "1"}}

In [ ]:
from langchain.messages import HumanMessage

input_msg1 = {
    "messages": [HumanMessage(content="강남역 근처 맛집 추천해줘.")],
    "context" : {}
}

response1 = app.invoke(input_msg1, config=config_1)

In [ ]:
response1

In [ ]:
# 현재 그래프 상태 확인 (무엇을 기다리는지)
snapshot = app.get_state(config_1)
if snapshot.next:
    print(f"\n⚠️ 그래프 상태: {snapshot.next} (Interrupt 대기 중)")
    print(f"❓ 질문 내용: {snapshot.tasks[0].interrupts[0].value}")

In [ ]:
# 사용자가 UI에서 "일식"이라고 입력했다고 가정



#### [Command](https://reference.langchain.com/python/langgraph/types/?h=command#langgraph.types.Command)

In [ ]:
result_2 = app.invoke(resume_command, config=config_1)

In [ ]:
result_2

In [ ]:
result_2['messages'][-1].content

## Interrupts in tools

### Model & Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [ ]:
from langchain.tools import tool

@tool
def ask_human(question: str) -> str:
    """
    사용자에게 추가 정보를 물어볼 때 사용하는 도구입니다.
    사용자의 답변을 받으려면 이 도구를 호출하세요.
    """

    # 이 함수 본체는 실행되지 않고, 아래 human_node에서 interrupt로 대체
    return "Human input required"

In [ ]:
tools = [ask_human]

### State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

### Node 정의

In [ ]:
# 메인 에이전트 노드
def agent_node(state: ChatState):
    llm_with_tools = model.bind_tools(tools)

    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
from langchain.messages import ToolMessage

def human_node(state: ChatState):
    """
    AI가 'ask_human' 도구를 호출했을 때 실행되는 노드입니다.
    실제로 도구를 실행하는 대신, interrupt를 걸어 사용자 입력을 받습니다.
    """
    # 1. AI가 요청한 도구 호출 정보(질문 내용) 가져오기
    last_message = state["messages"][-1]
    tool_call = last_message.tool_calls[0]
    question_to_user = tool_call["args"]["question"]

    # 2. [Dynamic Interrupt]


    # 3. 사용자의 답변을 '도구의 실행 결과(ToolMessage)'인 척 포장해서 return
    return {

    }

### 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("agent", agent_node)
workflow.add_node("human_node", human_node)

In [ ]:
# 조건부 함수
def should_continue(state: ChatState):
    last_message = state["messages"][-1]


In [ ]:
# 조건부 엣지
workflow.add_conditional_edges(
    "agent",
    should_continue,
    ["human_node", END]
)

In [ ]:
workflow.add_edge(START, "agent")
workflow.add_edge("human_node", "agent") # 답변을 받은 후 다시 에이전트로 돌아가서 생각하게 함

In [ ]:
memory = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=memory)

In [ ]:
app

In [ ]:
thread_config = {"configurable": {"thread_id": "2"}}

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

input_data = {
    "messages": [
        SystemMessage(content="당신은 맛집 추천 비서입니다. 위치와 메뉴를 모르면 반드시 도구를 호출해 물어보세요."),
        HumanMessage(content="맛집 추천 좀 해줄래?")
    ]
}

In [ ]:
result_1 = app.invoke(input_data, config=thread_config)

In [ ]:
# 현재 그래프 상태 확인 (무엇을 기다리는지)
snapshot = app.get_state(thread_config)
if snapshot.next:
    # interrupt에 담긴 질문 내용 확인
    # (실제 서비스에선 이걸 프론트엔드로 보냄)
    question = snapshot.tasks[0].interrupts[0].value

In [ ]:
from langgraph.types import Command

natural_language_answer = "음... 오늘은 비도 오고 하니까 따뜻한 우동이나 일식 같은 게 먹고 싶어."

# Command 객체를 사용하여 멈춘 지점에 값을 던져줌
resume_command = Command(resume=natural_language_answer)

In [ ]:
result_2

In [ ]:
result_2

In [ ]:
result_2['messages'][-1].content

## Validating Human Input

### Model & Tool 정의

### State 정의

### Node 정의

In [ ]:
# 2. 검증 로직이 포함된 Human Node
def human_validating_node(state: ChatState):
    # 1. AI가 던진 질문 가져오기
    last_message = state["messages"][-1]
    tool_call = last_message.tool_calls[0]
    original_question = tool_call["args"]["question"]

    # 2. 유효한 메뉴 목록 정의
    valid_categories = ["한식", "중식", "일식", "양식", "분식"]

    # 질문 텍스트 초기화
    current_prompt = original_question

    # [무한 루프] 제대로 된 답이 나올 때까지 가둔다.
    while True:
        # A. 멈추고 질문 (또는 에러 메시지와 함께 재질문)

        # B. 검증 로직 (Validation)
        # 입력값에 유효 카테고리가 포함되어 있는지 확인
        # (예: "매운 한식 먹을래" -> "한식" 포함 -> 통과)
        if any(category in user_input for category in valid_categories):

            # C. 검증 통과 시 ToolMessage 반환 (루프 탈출)
            return {
                "messages": [
                    ToolMessage(
                        content=user_input,
                        tool_call_id=tool_call["id"]
                    )
                ]
            }

        else:
            # D. 실패 시 프롬프트 수정 후 continue (다시 A로 돌아감)


In [ ]:
# 메인 에이전트 노드
def agent_node(state: ChatState):
    llm_with_tools = model.bind_tools(tools)

    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("agent", agent_node)
workflow.add_node("human_node", human_validating_node)

In [ ]:
# 조건부 함수
def should_continue(state: ChatState):
    last_message = state["messages"][-1]

    # 도구 호출이 있는데, 그게 'ask_human'이면 사람 노드로 보냄
    if last_message.tool_calls:
        if last_message.tool_calls[0]["name"] == "ask_human":
            return "human_node"

    # 도구 호출이 없으면 종료 (답변 완료)
    return END

In [ ]:
workflow.add_conditional_edges(
    "agent",
    should_continue,
    ["human_node", END]
)

In [ ]:
workflow.add_edge(START, "agent")
workflow.add_edge("human_node", "agent") # 답변을 받은 후 다시 에이전트로 돌아가서 생각하게 함

In [ ]:
app = workflow.compile(checkpointer=memory)

In [ ]:
app

In [ ]:
thread_config = {"configurable": {"thread_id": "3"}}

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

input_data = {
    "messages": [
        SystemMessage(content="당신은 맛집 추천 비서입니다. 위치와 메뉴를 모르면 반드시 도구를 호출해 물어보세요."),
        HumanMessage(content="맛집 추천 좀 해줄래?")
    ]
}

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

app.invoke(input_data, config=thread_config)

In [ ]:
# 현재 그래프 상태 확인 (무엇을 기다리는지)
snapshot = app.get_state(thread_config)
if snapshot.next:
    question = snapshot.tasks[0].interrupts[0].value
    print(f"\n⚠️ [Interrupt 발생] AI의 질문: {question}")

In [ ]:
app.invoke(Command(resume="신발 튀김"), config=thread_config)

In [ ]:
snapshot = app.get_state(thread_config)
print(f"⚠️ [재질문] 내용: {snapshot.tasks[0].interrupts[0].value}")

In [ ]:
result = app.invoke()

In [ ]:
result['messages'][-1].content

# Command with Goto

### Model & Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

### State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    selected_menu: str # 선택된 메뉴 저장

### Node 정의

In [ ]:
# [Node 1] 메뉴 추천
def recommender_node(state: ChatState):
    # 현재 상태(이전에 추천했던 메뉴)를 확인
    last_menu = state.get("selected_menu")

    # 만약 이전에 A코스를 추천했다면 -> 거절당하고 다시 온 것이므로 B코스 추천
    if last_menu == "A 코스 (20만원)":
        print("\n🍣 [Recommender] 그렇다면. 실속형 'B 코스 (8만원)'을 추천합니다.")
        return {"selected_menu": "B 코스 (8만원)"}

    # 기본(최초) 추천
    else:
        print("\n🍣 [Recommender] 쉐프 추천: 'A 코스 (20만원)'")
        return {"selected_menu": "A 코스 (20만원)"}

In [ ]:
# [Node 2] 승인/거절 및 경로 변경 (Router 역할 겸임)
def human_approval_node(state: ChatState):
    menu = state["selected_menu"]

    # 1. 멈추고 물어봄 (Interrupt)

    # 2. 답변에 따라 즉시 순간이동 (Command with goto)


In [ ]:
# [Node 3] 예약 확정 (승인 시 이동할 곳)
def booking_node(state: ChatState):
    menu = state["selected_menu"]
    print(f"\n✅ [Booking] '{menu}' 예약이 확정되었습니다! (문자 발송 완료)")
    return {"messages": [f"'{menu}' 예약 완료"]}

### 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)

workflow.add_node("recommender", recommender_node)
workflow.add_node("approval", human_approval_node)
workflow.add_node("booking", booking_node)

In [ ]:
workflow.add_edge(START, "recommender")
workflow.add_edge("recommender", "approval")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

In [ ]:
config = {"configurable": {"thread_id": "goto-demo"}}

In [ ]:
app.invoke({"messages": [HumanMessage(content="예약해줘")]}, config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"⚠️ 대기 중: {snapshot.tasks[0].interrupts[0].value}")

In [ ]:
app.invoke(Command(resume="no"), config=config)

In [ ]:
snapshot = app.get_state(config)
print(f"{snapshot.tasks[0].interrupts[0].value}")

In [ ]:
result = app.invoke(Command(resume="yes"), config=config)

In [ ]:
result['messages'][-1].content

# Command with Goto (실습)

### Model & Tool 정의

### State 정의

### Node 정의

### 그래프 생성

# Subgraph

### Model & Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

### State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ReservationState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    menu: str
    price: int
    status: str
    chef_name: str
    cooking_time: str

### Child Graph (주방 시스템)

In [ ]:
# 주방만의 별도 워크플로우를 정의
def check_ingredients(state: ReservationState):
    print(f"\n   🍳 [주방/재료팀] '{state['menu']}' 재료 재고 확인 중...")
    # (로직 생략: 무조건 재료가 있다고 가정)
    return {"status": "ok"}

In [ ]:
def assign_chef(state: ReservationState):
    menu = state['menu']
    if "오마카세" in menu:
        chef = "Master Jiro"
        time = "40분"
    else:
        chef = "Chef Kim"
        time = "20분"

    print(f"   👨‍🍳 [주방/인사팀] {chef} 쉐프 배정 완료 (예상 소요: {time})")
    return {"chef_name": chef, "cooking_time": time, "status": "cooking"}

In [ ]:
kitchen_builder = StateGraph(ReservationState)
kitchen_builder.add_node("check_stock", check_ingredients)
kitchen_builder.add_node("assign_chef", assign_chef)

kitchen_builder.add_edge(START, "check_stock")
kitchen_builder.add_edge("check_stock", "assign_chef")
kitchen_builder.add_edge("assign_chef", END)

In [ ]:
# 주방 그래프 컴파일 (이 자체가 하나의 노드처럼 쓰임)

kitchen_subgraph = kitchen_builder.compile()

In [ ]:
kitchen_subgraph

### Parent Graph (메인 홀 시스템)

In [ ]:
# [부서 1] 메뉴 추천
def menu_recommender(state: ReservationState):
    last_msg = state["messages"][-1].content if state["messages"] else ""

    if "비싸" in last_msg or "부담" in last_msg:
        proposal = "B코스 (런치 스페셜)"
        price = 80000
        msg = f"🍣 부담 없는 '{proposal} ({price:,}원)'은 어떠신가요?"
    else:
        proposal = "A코스 (쉐프 오마카세)"
        price = 150000
        msg = f"🍣 시그니처 '{proposal} ({price:,}원)'를 추천합니다."

    return {"menu": proposal, "price": price, "messages": [msg]}

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class UserIntent(BaseModel):
    action: Literal["confirm", "change", "cancel"] = Field(description="확정, 변경, 취소 판단")

In [ ]:
intent_analyzer = model.with_structured_output(UserIntent)

In [ ]:
# [부서 2] Agentic Router
def customer_confirm_node(state: ReservationState):
    menu = state['menu']
    price = state['price']

    user_input = interrupt(f"'{menu}' ({price:,}원)으로 하시겠습니까?")

    # LLM 의도 분석
    analysis = intent_analyzer.invoke(f"제안: {menu}, 답변: {user_input}")
    print(f"🧠 [AI Router]: 의도='{analysis.action}'")

    if analysis.action == "confirm":
        # 여기서 'kitchen'으로 보내면, 연결된 Subgraph가 실행됨
        return Command(

        )
    elif analysis.action == "change":
        return Command(

        )
    else:
        return Command(

        )

In [ ]:
# [부서 3] 취소 처리
def cancel_node(state: ReservationState):
    print(f"\n👋 [카운터]: 예약 종료.")
    return {"status": "cancelled"}

In [ ]:
parent_builder = StateGraph(ReservationState)

parent_builder.add_node("recommender", menu_recommender)
parent_builder.add_node("confirm", customer_confirm_node)
parent_builder.add_node("cancel", cancel_node)

In [ ]:
# 컴파일된 Subgraph를 '노드'로 등록


In [ ]:
# 엣지 등록

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
app = parent_builder.compile(checkpointer=checkpointer)

In [ ]:
app

### 실행

In [ ]:
config = {"configurable": {"thread_id": "subgraph-demo-1"}}

In [ ]:
app.invoke({"messages": []}, config=config)

In [ ]:
user_response = "좋아요, 그걸로 주세요."
result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
result

In [ ]:
print(f"메뉴: {result['menu']}")
print(f"상태: {result['status']}")
print(f"담당 쉐프: {result.get('chef_name')}")

In [ ]:
config = {"configurable": {"thread_id": "subgraph-demo-2"}}

In [ ]:
app.invoke({"messages": []}, config=config)

In [ ]:
user_response = "아니요.. 너무 비싸요."
result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
user_response = "그렇게 할게요."
final_result = app.invoke(Command(resume=user_response), config=config)

In [ ]:
final_result

In [ ]:
print(f"메뉴: {final_result['menu']}")
print(f"상태: {final_result['status']}")
print(f"담당 쉐프: {final_result.get('chef_name')} (Subgraph에서 생성된 정보!)")

# State 분리

## Child Graph

### State 정의

In [ ]:
from typing_extensions import TypedDict

class DeliveryState(TypedDict):


### Node 정의

In [ ]:
def pickup_package(state: DeliveryState):
    print(f"\n   🛵 [배달업체] 픽업 완료! (물품: {state['package_info']})")
    return {"delivery_status": "picked_up"}

In [ ]:
def deliver_to_customer(state: DeliveryState):
    print(f"   Hit [배달업체] {state['address']}로 배송 중... 띵동! 배달 완료.")
    return {"delivery_status": "delivered"}

### 그래프 생성

In [ ]:
delivery_builder = StateGraph(DeliveryState)

delivery_builder.add_node("pickup", pickup_package)
delivery_builder.add_node("deliver", deliver_to_customer)

delivery_builder.add_edge(START, "pickup")
delivery_builder.add_edge("pickup", "deliver")
delivery_builder.add_edge("deliver", END)

In [ ]:
# 하위 그래프 컴파일
delivery_graph = delivery_builder.compile()

In [ ]:
delivery_graph

## Parent Graph

### State 정의


In [ ]:
class RestaurantState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    menu: str
    customer_address: str
    final_status: str

### Node 정의

In [ ]:
def cook_food(state: RestaurantState):
    print(f"\n👨‍🍳 [주방] '{state['menu']}' 조리 완료! 포장했습니다.")
    return {"messages": ["조리 완료"]}

In [ ]:
# 외부 그래프를 호출하는 '브릿지(Bridge)' 노드
def call_delivery_service(state):
    print("\n📞 [매니저] 배달 업체 호출 중...")

    # 1. 내 공책(Parent)의 정보를 -> 외부 업체 양식(Child)으로 변환 (Mapping)

    # 2. 함수 안에서 그래프를 실행 (API 호출하듯이)

    # 3. 결과를 받아서 내 공책에 기록


### 그래프 생성

In [ ]:
parent_builder = StateGraph(RestaurantState)

parent_builder.add_node("kitchen", cook_food)
parent_builder.add_node("delivery_manager", call_delivery_service)

parent_builder.add_edge(START, "kitchen")
parent_builder.add_edge("kitchen", "delivery_manager")
parent_builder.add_edge("delivery_manager", END)

In [ ]:
app = parent_builder.compile()

In [ ]:
app

### 실행

In [ ]:
inputs = {
    "menu": "특선 초밥 세트",
    "customer_address": "서울시 강남구 역삼동 123",
    "messages": []
}

In [ ]:
result = app.invoke(inputs)

In [ ]:
result

# Subgraph (실습)

# 📝 [실습 1] "여행용 환전 지갑" 만들기

### 배경: 당신은 여행 가이드 앱을 만들고 있습니다. 사용자가 "100달러 환전해줘"라고 말하면, 외부 은행 시스템(Subgraph)에 요청하여 환전을 처리해야 합니다.

### 제약 사항:
1. 여행 앱(Parent)과 은행(Child)은 서로 다른 보안 시스템을 쓰기 때문에 상태(State)를 공유하지 않습니다.

2. 따라서 add_node(subgraph)를 쓸 수 없고, 노드 함수 안에서 invoke()를 사용해야 합니다.

3. Parent State의 request_amount(요청 금액)를 Child State의 usd_amount(입금액)로 변환해서 넘겨줘야 합니다.

### ✅ 요구사항:

1. 빈칸으로 비워둔 exchange_bridge_node 함수를 완성하세요.

2. Parent의 request_amount 값을 읽어옵니다.

3. Child(bank_graph)를 invoke하여 환전을 수행합니다.

4. 결과로 받은 krw_result를 Parent의 balance_krw에 업데이트하세요.

In [ ]:
# @title
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. [Child] 외부 은행 시스템 (수정 금지)
# ==========================================
class BankState(TypedDict):
    usd_amount: int   # 입금된 달러
    krw_result: int   # 환전된 원화
    receipt: str      # 영수증

def calculate_exchange(state: BankState):
    rate = 1400 # 환율 1400원 가정
    krw = state['usd_amount'] * rate
    print(f"   🏦 [은행] ${state['usd_amount']} -> {krw}원 환전 처리 중...")
    return {
        "krw_result": krw,
        "receipt": f"[영수증] {state['usd_amount']} USD -> {krw} KRW"
    }

bank_builder = StateGraph(BankState)
bank_builder.add_node("calc", calculate_exchange)
bank_builder.add_edge(START, "calc")
bank_builder.add_edge("calc", END)
bank_graph = bank_builder.compile()

# ==========================================
# 2. [Parent] 여행 가이드 앱
# ==========================================
class TravelState(TypedDict):
    request_amount: int  # 사용자가 요청한 달러
    balance_krw: int     # 나의 원화 지갑 잔액
    status: str

# 사용자 요청을 처리하는 노드
def request_node(state: TravelState):
    amount = state['request_amount']
    print(f"\n✈️ [가이드] ${amount} 환전을 위해 은행에 연결합니다.")
    return {"status": "requesting"}

# ★ [미션] 여기가 문제입니다! ★
def exchange_bridge_node(state: TravelState):
    """
    1. Parent State에서 'request_amount'를 가져오세요.
    2. Bank State 양식({'usd_amount': ...})에 맞춰 데이터를 만드세요.
    3. bank_graph를 invoke() 하세요.
    4. 결과('krw_result')를 가져와서 Parent State의 'balance_krw'에 넣어서 반환하세요.
    """
    # (여기에 코드를 작성하세요)
    pass

# ==========================================
# 3. 그래프 조립 (수정 금지)
# ==========================================
app_builder = StateGraph(TravelState)
app_builder.add_node("guide", request_node)
app_builder.add_node("bank_bridge", exchange_bridge_node)

app_builder.add_edge(START, "guide")
app_builder.add_edge("guide", "bank_bridge")
app_builder.add_edge("bank_bridge", END)

app = app_builder.compile()

# 실행
print("### 환전 요청 실행 ###")
result = app.invoke({"request_amount": 100, "balance_krw": 0, "status": "start"})

print(f"\n🎉 [최종 결과] 내 지갑 잔액: {result['balance_krw']}원")

In [ ]:
# @title
# ★ [정답] ★
def exchange_bridge_node(state: TravelState):
    # 1. 내 공책(Parent)에서 정보 꺼내기
    my_request = state['request_amount']

    # 2. 외부 업체 공책(Child) 양식에 맞춰 데이터 포장 (Mapping)
    # Parent의 'request_amount' -> Child의 'usd_amount'
    bank_input = {"usd_amount": my_request}

    # 3. 외부 그래프 호출 (Invoke)
    # invoke()를 쓰면 해당 그래프가 독립적으로 실행되고 결과만 리턴됩니다.
    bank_output = bank_graph.invoke(bank_input)

    # 4. 결과를 받아서 내 공책에 기록
    # Child의 'krw_result' -> Parent의 'balance_krw'
    exchanged_money = bank_output["krw_result"]
    print(f"✈️ [가이드] 은행에서 {exchanged_money}원을 받아왔습니다.")

    return {
        "balance_krw": exchanged_money,
        "status": "completed"
    }

# 📝 [실습 2] AI 뉴스레터 편집국

목표: 주제만 던지면 기획, 검색, 작성, 그리고 **인간 편집장의 검수(Review)**까지 거쳐 최종 원고를 발행하는 시스템 구축.

## 📋 프로젝트 명세서 (Requirements)
여러분이 구현해야 할 시스템의 요구사항은 다음과 같습니다.

### 구조 (Architecture):

- Parent Graph (편집국): 전체 흐름을 관장하며, **인간 편집장(Human)**과의 소통을 담당합니다.

- Subgraph (제작팀): **검색(Researcher)**과 **집필(Writer)**을 담당하는 에이전트들이 모여 있습니다.

### 워크플로우 (Workflow):

1. 사용자가 주제(Topic)를 입력합니다.

2. **제작팀(Subgraph)**이 작동하여 초안을 작성합니다.

3. **편집장(Parent)**에게 Interrupt가 걸립니다. 초안을 보여주고 승인을 요청합니다.

4. 승인(Approve) 시: 최종 발행하고 종료합니다.

5. 거절(Reject) 시: 피드백(Feedback)을 입력받아 다시 제작팀(Subgraph)으로 돌려보냅니다. (Command 사용)

6. 수정(Revision): 제작팀은 피드백을 반영하여 원고를 수정합니다. (State 유지)

In [ ]:
# @title
from typing import Annotated, TypedDict, List, Literal, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# ====================================================
# 1. State 정의 (Shared State)
# ====================================================
class NewsState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str
    research_data: Optional[str]
    draft: Optional[str]
    feedback: Optional[str]
    status: str

# [요청하신 모델 적용]
# 실제 실행 시 API Key가 필요하며, gpt-5-nano가 없다면 gpt-4o-mini 등으로 대체될 수 있습니다.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# ====================================================
# 2. [Subgraph] 콘텐츠 제작팀 (Real AI 적용)
# ====================================================

def researcher_node(state: NewsState):
    topic = state["topic"]
    feedback = state.get("feedback")

    print(f"\n🔎 [Researcher] AI가 '{topic}' 자료를 조사하고 있습니다...")

    # 1. 피드백 유무에 따른 프롬프트 구성
    if feedback:
        query = f"주제: '{topic}'\n추가 지시사항(피드백): {feedback}\n\n위 내용을 반영하여 핵심 정보를 조사하세요."
        print(f"   ↳ (지시사항 반영 중: {feedback})")
    else:
        query = f"주제: '{topic}'\n\n위 주제에 대한 최신 트렌드와 핵심 사실 3가지를 요약해서 조사하세요."

    # 2. LLM 호출 (Real Generation)
    response = llm.invoke([
        SystemMessage(content="당신은 세계 최고의 IT 트렌드 리서처입니다. 팩트 기반으로 명확하게 요약하세요."),
        HumanMessage(content=query)
    ])

    return {"research_data": response.content}

def writer_node(state: NewsState):
    print(f"\n✍️ [Writer] AI가 뉴스레터 초안을 작성하고 있습니다...")

    data = state["research_data"]
    feedback = state.get("feedback")

    # 1. 프롬프트 구성
    if feedback:
        prompt = f"""
        [지시사항]
        편집장의 피드백: "{feedback}"

        위 피드백을 철저히 반영하여, 아래 조사 자료를 바탕으로 뉴스레터를 '다시' 작성하세요.

        [조사 자료]
        {data}
        """
    else:
        prompt = f"""
        [지시사항]
        아래 조사 자료를 바탕으로 독자들이 흥미를 느낄만한 뉴스레터 초안을 작성하세요.
        서론-본론-결론 구조를 갖추세요.

        [조사 자료]
        {data}
        """

    # 2. LLM 호출
    response = llm.invoke([
        SystemMessage(content="당신은 베스트셀러 작가이자 뉴스레터 에디터입니다. 전문적이면서도 읽기 쉬운 톤을 유지하세요."),
        HumanMessage(content=prompt)
    ])

    return {"draft": response.content}

# Subgraph 조립
content_builder = StateGraph(NewsState)
content_builder.add_node("researcher", researcher_node)
content_builder.add_node("writer", writer_node)

content_builder.add_edge(START, "researcher")
content_builder.add_edge("researcher", "writer")
content_builder.add_edge("writer", END)

content_team = content_builder.compile()

# ====================================================
# 3. [Parent Graph] 편집국 (Human-in-the-loop)
# ====================================================

def human_review_node(state: NewsState):
    current_draft = state["draft"]

    print(f"\n🛑 [Editor-in-Chief] 초안 검수 (Interrupt)")
    print("=" * 60)
    print(f"[AI가 작성한 초안]\n\n{current_draft}")
    print("=" * 60)

    # 인간의 개입
    review_input = interrupt("승인하려면 'ok', 수정하려면 '피드백'을 입력하세요.")

    print(f"▶️ [Editor Decision]: {review_input}")

    # 라우팅
    if review_input.lower() in ["ok", "승인", "yes"]:
        return Command(
            update={
                "status": "published",
                "messages": [HumanMessage(content="편집장 승인 완료")]
            },
            goto="publisher"
        )
    else:
        return Command(
            update={
                "feedback": review_input,
                "messages": [HumanMessage(content=f"피드백: {review_input}")]
            },
            goto="content_team" # 다시 Subgraph로 보냄
        )

def publisher_node(state: NewsState):
    print(f"\n🚀 [Publisher] 뉴스레터 발행 완료! (구독자에게 전송됨)")
    return {"status": "completed"}

# Parent Graph 조립
parent_builder = StateGraph(NewsState)

parent_builder.add_node("content_team", content_team) # Subgraph를 노드로!
parent_builder.add_node("reviewer", human_review_node)
parent_builder.add_node("publisher", publisher_node)

parent_builder.add_edge(START, "content_team")
parent_builder.add_edge("content_team", "reviewer")
parent_builder.add_edge("publisher", END)

checkpointer = InMemorySaver()
app = parent_builder.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "newsletter-prod-1"}}

app.invoke({"topic": "2025년 생성형 AI 트렌드"}, config=config)


🔎 [Researcher] AI가 '2025년 생성형 AI 트렌드' 자료를 조사하고 있습니다...

✍️ [Writer] AI가 뉴스레터 초안을 작성하고 있습니다...

🛑 [Editor-in-Chief] 초안 검수 (Interrupt)
[AI가 작성한 초안]

제목: 2025년 생성형 AI 트렌드 3대 축과 실무 로드맵

서론
AI의 속도에 발맞추려면 기술 그 자체보다 먼저 검토해야 할 것이 있습니다. 바로 거버넌스의 표준화, 멀티모달 코파일럿의 생산성 도구화, 그리고 엣지/오픈소스 중심의 데이터 주권 강화입니다. 오늘의 뉴스레터는 이 세 가지 축을 바탕으로, 기업이 안전하고 효과적으로 AI를 도입하고 운영하는 데 필요한 실무 가이드와 우선순위를 제시합니다. 각 축은 서로를 보완하며, 단계적으로 도입할 때 가장 큰 ROI를 기대할 수 있습니다.

본론
핵심 사실 1. 엔터프라이즈 거버넌스와 컴플라이언스의 표준화 가속
- 요지: 기업이 생성형 AI를 안정적으로 활용하기 위한 거버넌스 체계가 표준화되고 있습니다.
- 구체 내용 요약: 데이터 소스 추적, 정책 기반 프롬프트 거버넌스, 자동 위험 평가 및 감사 로깅이 기본 구성요소로 자리잡습니다.
- 기대 효과: 데이터 사용 규정 준수 강화, 안전성 향상, 감사 가능성 증가로 기업 신뢰도와 ROI 상승.
- 실무 시사점(우선 검토 항목): 데이터 거버넌스 정책 수립, 로깅/감사 체계 구축, 프롬프트 관리 자동화 도구 도입을 먼저 고려하세요.
- 적용 팁: 기업 내 데이터 출처를 한눈에 추적할 수 있는 기록 체계와, 프롬프트 정책의 버전 관리 프로세스를 빠르게 시도해 보십시오.

핵심 사실 2. 멀티모달 코파일럿의 생산성 도구화 확산
- 요지: 텍스트, 이미지, 음성, 코드, 비디오를 한꺼번에 다루는 멀티모달 모델이 생산성 도구의 핵심 코파일럿으로 확산됩니다.
- 구체 내용 요약: 디자인, 마케팅, 소프트웨어 개발, 고객지원 등 다양한 업무 흐름에서 멀티모달 입력/출력을 활용한 자동 생성·요약·전환이 통합된 워크플

{'messages': [],
 'topic': '2025년 생성형 AI 트렌드',
 'research_data': '다음은 2025년 생성형 AI 트렌드에 대한 최신 관찰과 예측을 바탕으로 정리한 3가지 핵심 사실입니다.\n\n- 핵심 사실 1: 엔터프라이즈 거버넌스와 컴플라이언스의 표준화 가속\n  - 요지: 기업에서 생성형 AI를 업무에 안정적으로 도입하기 위한 거버넌스 체계가 표준화되고 있습니다.\n  - 구체 내용: 데이터 소스 추적, 프롬프트 거버넌스(정책 기반 관리), 자동 위험 평가 및 감사 로깅 기능이 기업용 AI 플랫폼의 기본 구성요소로 자리잡습니다.\n  - 기대 효과: 데이터 사용 규정 준수, 안전성 강화, 감사 가능성 향상으로 기업 신뢰도와 ROI 상승에 기여합니다.\n  - 실무 시사점: 데이터 거버넌스 정책 수립과 로깅/감사 체계 구축, 프롬프트 관리 자동화 도구 도입을 우선 검토하세요.\n\n- 핵심 사실 2: 멀티모달 코파일럿의 생산성 도구화 확산\n  - 요지: 텍스트, 이미지, 음성, 코드, 비디오를 한꺼번에 다루는 멀티모달 모델이 기업 생산성 도구의 핵심 코파일럿으로 보급될 가능성이 큽니다.\n  - 구체 내용: 디자인, 마케팅, 소프트웨어 개발, 고객지원 등 다양한 업무 흐름에서 멀티모달 특성을 활용한 자동 생성·요약·전환 작업이 통합된 워크플로우를 지원합니다.\n  - 기대 효과: 창의적 생산성 향상, 컨텍스트 간 전환의 효율성 증가, 반복 작업의 자동화로 인한 운영비용 절감.\n  - 실무 시사점: 멀티모달 입력/출력 파이프라인을 기존 도구에 연결하고, 코파일럿 기반의 업무 프로세스 개선을 우선 시도해 보세요.\n\n- 핵심 사실 3: 엣지/오픈소스 중심의 경량화 및 데이터 주권 강화\n  - 요지: 데이터 프라이버시와 비용 효율성 요구가 커지면서 로컬 처리와 오프-클라우드 배포를 지원하는 경량 모델의 채택이 늘고 있습니다.\n  - 구체 내용: 경량화된 파인튜닝 가능 모델, 양방향은 하이브리드 배포, 엣지 디바이스에서

In [ ]:
snapshot = app.get_state(config)
print(f"\n⚠️ 현재 단계: {snapshot.next}")

In [ ]:
feedback_msg = "너무 딱딱해요. 좀 더 유머러스하고 친근한 말투로 바꿔주세요."

app.invoke(Command(resume=feedback_msg), config=config)


🛑 [Editor-in-Chief] 초안 검수 (Interrupt)
[AI가 작성한 초안]

제목: 2025년 생성형 AI 트렌드 3대 축과 실무 로드맵

서론
AI의 속도에 발맞추려면 기술 그 자체보다 먼저 검토해야 할 것이 있습니다. 바로 거버넌스의 표준화, 멀티모달 코파일럿의 생산성 도구화, 그리고 엣지/오픈소스 중심의 데이터 주권 강화입니다. 오늘의 뉴스레터는 이 세 가지 축을 바탕으로, 기업이 안전하고 효과적으로 AI를 도입하고 운영하는 데 필요한 실무 가이드와 우선순위를 제시합니다. 각 축은 서로를 보완하며, 단계적으로 도입할 때 가장 큰 ROI를 기대할 수 있습니다.

본론
핵심 사실 1. 엔터프라이즈 거버넌스와 컴플라이언스의 표준화 가속
- 요지: 기업이 생성형 AI를 안정적으로 활용하기 위한 거버넌스 체계가 표준화되고 있습니다.
- 구체 내용 요약: 데이터 소스 추적, 정책 기반 프롬프트 거버넌스, 자동 위험 평가 및 감사 로깅이 기본 구성요소로 자리잡습니다.
- 기대 효과: 데이터 사용 규정 준수 강화, 안전성 향상, 감사 가능성 증가로 기업 신뢰도와 ROI 상승.
- 실무 시사점(우선 검토 항목): 데이터 거버넌스 정책 수립, 로깅/감사 체계 구축, 프롬프트 관리 자동화 도구 도입을 먼저 고려하세요.
- 적용 팁: 기업 내 데이터 출처를 한눈에 추적할 수 있는 기록 체계와, 프롬프트 정책의 버전 관리 프로세스를 빠르게 시도해 보십시오.

핵심 사실 2. 멀티모달 코파일럿의 생산성 도구화 확산
- 요지: 텍스트, 이미지, 음성, 코드, 비디오를 한꺼번에 다루는 멀티모달 모델이 생산성 도구의 핵심 코파일럿으로 확산됩니다.
- 구체 내용 요약: 디자인, 마케팅, 소프트웨어 개발, 고객지원 등 다양한 업무 흐름에서 멀티모달 입력/출력을 활용한 자동 생성·요약·전환이 통합된 워크플로우를 지원합니다.
- 기대 효과: 창의적 생산성 증가, 컨텍스트 간 전환의 효율성 증가, 반복 작업의 자동화로 운영비용 절감.
- 실무 시사점(우선 시도 포인트): 멀티

{'messages': [HumanMessage(content='피드백: 너무 딱딱해요. 좀 더 유머러스하고 친근한 말투로 바꿔주세요.', additional_kwargs={}, response_metadata={}, id='eac8f305-b70a-4045-9b90-99011859d0a4')],
 'topic': '2025년 생성형 AI 트렌드',
 'research_data': "좋아요! 2025년 생성형 AI 트렌드를 팩트 중심으로, 그런데 조금 친근하고 유머 섞인 톤으로 정리해볼게요.\n\n주요 핵심 포인트 10가지\n\n1) 멀티모달의 일반화\n- 한 모델이 텍스트+이미지+음성+코드+데이터까지 한꺼번에 다룬다. 대화 중에 사진을 분석하고, 디자인 피드백은 즉시 그림으로 보여주는 식.\n- 비즈니스 포인트: 콘텐츠 생산 속도↑, 사용자 편의성 대폭 향상.\n\n2) AI 코파일럿의 보편화\n- 개발자, 디자이너, 마케터 등 다양한 직군의 '삐딱한 친구'가 돼주는 코파일럿이 대세. IDE에서 코드 제안, 문서 초안 작성, 데이터 분석까지 도와줌.\n- 비즈니스 포인트: 시간 절약과 오류 감소.\n\n3) 에이전트 기반 자동화의 상용화\n- AI가 일을 스스로 기획하고 실행하는 에이전트가 워크플로를 연결. 예를 들면 데이터 파이프라인 설계→실행→결과 보고까지 한 번에 처리.\n- 비즈니스 포인트: 반복 업무 최소화, 운영 효율 증대.\n\n4) 합성 데이터와 데이터 프라이버시 중심 전략\n- 실데이터 대신 합성 데이터를 많이 활용하고, 차등 프라이버시나 데이터 라이센스 관리와 함께 운용.\n- 비즈니스 포인트: 개인정보 규정 준수와 데이터 편향 관리가 쉬워짐.\n\n5) 엣지/온디바이스 실행의 확대\n- 네트워크 연결이 없어도 동작하고, 지연도 낮추고, 프라이버시도 보호하는 방향으로 강화.\n- 비즈니스 포인트: 모바일/현장 사용 사례에서의 신뢰성 높임.\n\n6) 효율성과 친환경 AI\n- 모델 경량화, 양자화, 희소화(sparsity) 등으로 비용과 에

In [ ]:
result = app.invoke(Command(resume="ok"), config=config)


🛑 [Editor-in-Chief] 초안 검수 (Interrupt)
[AI가 작성한 초안]

안녕하세요, 팀 여러분! 오늘은 2025년 생성형 AI 트렌드를 팩트는 팩트대로, 분위기는 살짝 친근하게 풀어봤어요. 딱딱한 용어는 닫아두고, 현장에서 바로 써먹을 포인트 위주로 정리했습니다. 자, 시작합니다.

1) 멀티모달의 일반화
- 한 모델이 텍스트, 이미지, 음성, 코드, 데이터까지 한꺼번에 다룬다. 대화 중 사진 분석하고, 디자인 피드백은 즉시 그림으로 보여주는 식.
- 비즈니스 포인트: 콘텐츠 생산 속도가 빨라지고, 사용자가 체감하는 편의성 대폭 상승.

2) AI 코파일럿의 보편화
- 개발자·디자이너·마케터 등 다양한 직군의 ‘삐딱한 친구’가 되어주는 코파일럿이 대세. IDE에서 코드 제안, 문서 초안 작성, 데이터 분석까지 도와줌.
- 비즈니스 포인트: 시간 절약과 오류 감소로 팀의 생산력 상승.

3) 에이전트 기반 자동화의 상용화
- AI가 일을 스스로 기획하고 실행하는 에이전트가 워크플로를 연결. 데이터 파이프라인 설계→실행→결과 보고까지 한 번에 처리.
- 비즈니스 포인트: 반복 업무 최소화, 운영 효율 대폭 개선.

4) 합성 데이터와 데이터 프라이버시 중심 전략
- 실데이터 대신 합성 데이터를 많이 활용하고, 차등 프라이버시나 데이터 라이선스 관리와 함께 운용.
- 비즈니스 포인트: 개인정보 규정 준수와 데이터 편향 관리가 쉬워짐.

5) 엣지/온디바이스 실행의 확대
- 네트워크 연결이 없어도 작동하고, 지연도 낮추고, 프라이버시도 보호하는 방향으로 강화.
- 비즈니스 포인트: 모바일/현장 사용 사례에서의 신뢰성 증가.

6) 효율성과 친환경 AI
- 모델 경량화, 양자화, 희소화(sparsity) 등으로 비용과 에너지 소비를 줄이는 기술이 표준으로 자리 잡음.
- 비즈니스 포인트: TCO 절감과 함께 그린 AI 이미지도 함께 향상.

7) 안전성, 거버넌스, 규제 대응 강화
- 평가 도구, 모델 카드 및 안전 가드레일, 워터마킹 등 투명성

# Multi-Agent (Supervisor - 중앙 집중형)
[공식 문서](https://docs.langchain.com/oss/python/langchain/multi-agent)

## Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

## State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class TeamState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
# 감독관이 내릴 수 있는 결정 목록
class RouteResponse(BaseModel):
    next_step: Literal[  "FINISH"] = Field(
        description="다음 작업을 수행할 팀원을 선택하거나, 모든 작업이 완료되었으면 FINISH를 선택"
    )
    reason: str = Field(description="선택한 이유")

In [ ]:
supervisor_agent = model.with_structured_output(RouteResponse)

## Node 정의

### Supervisor Node

In [ ]:
def supervisor_node(state: TeamState):
    print("\n👔 [Supervisor] 다음 작업자 결정 중...")

    # 1. 현재까지의 대화 내용을 보고 판단
    system_prompt = """
    당신은 블로그 작성 팀의 관리자입니다.
    1. 사용자 요청이 들어오면 'researcher'에게 조사를 시키세요.
    2. 조사 결과가 나오면 'writer'에게 글 작성을 시키세요.
    3. 글 작성이 완료되면 'FINISH'를 선언하세요.
    """

    # LLM에게 판단 요청
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    decision = supervisor_agent.invoke(messages)

    # 2. 판단 결과에 따라 라우팅 (Command Goto)


### Worker Node

In [ ]:
# [팀원 1] 검색 담당 (Researcher)
def researcher_node(state: TeamState):
    print("\n🔎 [Researcher] LLM이 자료를 조사/생성 중입니다...")

    # 시스템 프롬프트 부여
    system_msg = SystemMessage(content="""
    당신은 IT 트렌드 전문 리서처입니다.
    사용자의 요청 주제에 대해 핵심 트렌드, 기술 용어, 통계 등을 포함하여
    전문적인 조사 보고서를 3줄 요약 형태로 작성하세요.
    """)

    # 이전 대화 내용 + 내 역할(System)을 합쳐서 LLM에게 전달
    # state["messages"] 안에 사용자의 요청이 들어있음
    messages = [system_msg] + state["messages"]

    # LLM 실행
    response = model.invoke(messages)

    # 결과 반환 (name을 지정해야 감독관이 누가 말했는지 앎)


In [ ]:
# [팀원 2] 집필 담당 (Writer)
def writer_node(state: TeamState):
    print("\n✍️ [Writer] LLM이 블로그 포스팅을 작성 중입니다...")

    system_msg = SystemMessage(content="""
    당신은 테크 블로그 전문 작가입니다.
    위의 대화 내역에 있는 'Researcher'의 조사 결과를 바탕으로
    매력적인 블로그 포스팅(제목+본문)을 작성하세요.
    이모지를 적절히 사용하여 가독성을 높이세요.
    """)

    # 공유된 대화 기록(Researcher의 결과 포함)을 다 같이 넘겨줌
    messages = [system_msg] + state["messages"]

    response = model.invoke(messages)


## 그래프 생성

In [ ]:
workflow = StateGraph(TeamState)

workflow.add_node("supervisor", supervisor_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)

workflow.add_edge(START, "supervisor")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

In [ ]:
config = {"configurable": {"thread_id": "team-blog-1"}}

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="AI 최신 트렌드에 대해 블로그 글 써줘")]},
    config=config
)

새로운 작업자 '검수자'를 추가한다면 어떻게 구현하면 될까?

# Multi-Agent (Handoff (Network))
[LangGraph 공식 문서](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs)

[OpenAI Handoff](https://openai.github.io/openai-agents-python/handoffs/)

## Model&Tool 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

[tool runtime](https://reference.langchain.com/python/langgraph/agents/?h=toolruntime#langgraph.prebuilt.tool_node.InjectedStore)

In [ ]:
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime

@tool
def transfer_to_tech_support(runtime: ToolRuntime):
    """기술적인 문제(컴퓨터, 코드, 에러 등)를 해결해야 할 때 이 도구를 사용하여 기술팀으로 연결하세요."""



In [ ]:
@tool
def transfer_to_billing(runtime: ToolRuntime):
    """환불, 결제, 금액 관련 문제를 해결해야 할 때 이 도구를 사용하여 환불팀으로 연결하세요."""


In [ ]:
@tool
def transfer_to_reception(runtime: ToolRuntime):
    """일반적인 문의거나, 인사가 끝났으면 안내 데스크로 다시 연결하세요."""


## State 정의

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
      messages: Annotated[list[AnyMessage], add_messages]

## Node 정의

### Agent Nodes

In [ ]:
# [상담원 1] 안내 데스크 (시작점)
def receptionist_node(state: AgentState):
    print("\n💁 [Reception] 안녕하세요.")

    system_prompt = """
    당신은 고객센터의 친절한 안내 데스크입니다.
    1. 고객의 말을 듣고 인사하세요.
    2. '기술 문제'라면 tech_support 도구를 쓰세요.
    3. '돈/환불 문제'라면 billing 도구를 쓰세요.
    4. 그 외 잡담은 직접 답변하세요.
    """

    # 안내 데스크는 기술팀과 환불팀으로 보낼 수 있음
    tools = [transfer_to_tech_support, transfer_to_billing]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:
# [상담원 2] 기술 지원팀
def tech_node(state: AgentState):
    print("\n🛠️ [Tech Support] 기술적인 문제를 해결해드립니다.")

    system_prompt = """
    당신은 베테랑 기술 지원 엔지니어입니다.
    컴퓨터 고장, 파이썬 코드 에러 등을 해결해줍니다.
    만약 고객이 '환불'을 요구하면 즉시 billing 도구를 써서 넘기세요.
    해결이 끝나면 reception으로 넘기세요.
    """

    # 기술팀은 환불팀이나 안내 데스크로 보낼 수 있음
    tools = [transfer_to_billing, transfer_to_reception]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:
# [상담원 3] 환불팀
def billing_node(state: AgentState):
    print("\n💰 [Billing] 결제 및 환불 담당입니다.")

    system_prompt = """
    당신은 환불 담당자입니다.
    고객의 환불 요청을 처리해줍니다.
    기술적인 질문을 하면 tech_support로 넘기세요.
    해결이 끝나면 reception으로 넘기세요.
    """

    # 환불팀은 기술팀이나 안내 데스크로 보낼 수 있음
    tools = [transfer_to_tech_support, transfer_to_reception]
    model_with_tools = model.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}

## 그래프 생성

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("receptionist", receptionist_node)
workflow.add_node("tech_support", tech_node)
workflow.add_node("billing", billing_node)

In [ ]:
from langgraph.prebuilt import ToolNode

# 도구를 실행할 노드 (ToolNode)
# 에이전트가 도구를 호출하면 -> Tool Node로 가서 -> 함수 실행 -> Command 반환 -> 다른 노드로 점프
all_tools = [transfer_to_tech_support, transfer_to_billing, transfer_to_reception]
tool_node = ToolNode(all_tools)

In [ ]:
tool_node

In [ ]:
workflow.add_node("tools", tool_node)

In [ ]:
# 시작은 안내 데스크
workflow.add_edge(START, "receptionist")

In [ ]:
# 모든 노드는 tools 또는 END 노드 중 하나 (조건부 엣지)


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
app

In [ ]:
config = {"configurable": {"thread_id": "handoff-demo-1"}}

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="제 컴퓨터 화면이 안 나와요. 고장난 것 같아요.")]},
    config=config
)

In [ ]:
app.invoke(
    {"messages": [HumanMessage(content="아직 안 되는데, 그냥 환불하고 싶어요.")]},
    config=config
)